# Fase 2 — Tratamento da camada Silver

Objetivo: partir da tabela Bronze (dado bruto) e produzir uma tabela **Silver**:
tipos corretos, valores limpos, e registros problemáticos **sinalizados** — não
descartados. Quem decide o que fazer com eles é a análise (Fase 4), não o
pipeline.

In [0]:
spark.sql("USE CATALOG fauna")
spark.sql("USE SCHEMA monitoramento")

df_bronze = spark.table("fauna.monitoramento.bronze_registros")
print(f"Lendo bronze_registros: {df_bronze.count()} registros")
df_bronze.printSchema()

## Corrigir tipos: temperatura, umidade e data

In [0]:
from pyspark.sql import functions as F

df_tipado = (
    df_bronze
    .withColumn(
        "temperatura_c",
        F.regexp_extract(F.col("temperatura_c"), r"(-?\d+\.?\d*)", 1).cast("double")
    )
    .withColumn(
        "umidade_pct",
        F.regexp_extract(F.col("umidade_pct"), r"(-?\d+\.?\d*)", 1).cast("double")
    )
    .withColumn(
        "data_hora_inicio",
        F.to_timestamp(F.col("data_hora_inicio"), "yyyy-MM-dd HH:mm:ss")
    )
)

df_tipado.select("temperatura_c", "umidade_pct", "data_hora_inicio").printSchema()
df_tipado.select("temperatura_c", "umidade_pct", "data_hora_inicio").show(5, truncate=False)

## Checar nulos gerados pela conversão

In [0]:
colunas_convertidas = ["temperatura_c", "umidade_pct", "data_hora_inicio"]

for coluna in colunas_convertidas:
    nulos_antes = df_bronze.filter(F.col(coluna).isNull()).count()
    nulos_depois = df_tipado.filter(F.col(coluna).isNull()).count()
    status = "✅" if nulos_depois == nulos_antes else "⚠️"
    print(f"{status} {coluna}: nulos na Bronze = {nulos_antes} | nulos após conversão = {nulos_depois}")

## Checagens de qualidade e domínio (Bônus 1)

In [0]:
cameras_validas = [f"CAM{str(i).zfill(2)}" for i in range(1, 13)]

df_flags = (
    df_tipado
    .withColumn("_flag_camera_invalida", ~F.col("id_camera").isin(cameras_validas))
    .withColumn("_flag_individuos_invalido", (F.col("individuos") < 1) | F.col("individuos").isNull())
    .withColumn("_flag_duracao_invalida", (F.col("duracao_segundos") <= 0) | F.col("duracao_segundos").isNull())
    .withColumn("_flag_temperatura_ausente", F.col("temperatura_c").isNull())
    .withColumn(
        "_flag_temperatura_fora_faixa",
        F.col("temperatura_c").isNotNull() & ((F.col("temperatura_c") < -5) | (F.col("temperatura_c") > 45))
    )
    .withColumn("_flag_umidade_ausente", F.col("umidade_pct").isNull())
    .withColumn(
        "_flag_umidade_fora_faixa",
        F.col("umidade_pct").isNotNull() & ((F.col("umidade_pct") < 0) | (F.col("umidade_pct") > 100))
    )
)

total = df_flags.count()
print(f"Total avaliado: {total}\n")

for flag in [
    "_flag_camera_invalida", "_flag_individuos_invalido", "_flag_duracao_invalida",
    "_flag_temperatura_ausente", "_flag_temperatura_fora_faixa",
    "_flag_umidade_ausente", "_flag_umidade_fora_faixa",
]:
    qtd = df_flags.filter(F.col(flag)).count()
    print(f"{flag}: {qtd} registros ({qtd/total*100:.2f}%)")

## Checar duplicidade real

In [0]:
from pyspark.sql.window import Window

janela_chave = ["id_camera", "data_hora_inicio", "especie"]

df_flags = df_flags.withColumn(
    "_flag_duplicado",
    F.count("*").over(Window.partitionBy(*janela_chave)) > 1
)

qtd_duplicados = df_flags.filter(F.col("_flag_duplicado")).count()
print(f"_flag_duplicado: {qtd_duplicados} registros ({qtd_duplicados/total*100:.2f}%)")

## Inspecionar os registros duplicados

In [0]:
df_flags.filter(F.col("_flag_duplicado")) \
    .orderBy("id_camera", "data_hora_inicio", "especie") \
    .select(
        "id_registro", "id_camera", "data_hora_inicio", "especie",
        "duracao_segundos", "individuos", "temperatura_c", "umidade_pct"
    ) \
    .show(20, truncate=False)

## Decisão de tratamento (Bônus 1)

Nenhum registro é descartado na Silver. Resumo do que foi encontrado e a
decisão tomada em cada caso:

- **Câmeras, `individuos`, `duracao_segundos`**: nenhuma inconsistência
  encontrada (0% em todas as flags).
- **`temperatura_c` ausente em 247 registros (4,94%) e `umidade_pct` ausente
  em 175 registros (3,50%)**: falha real de sensor, não erro de conversão
  (confirmado comparando nulos antes/depois do cast). Decisão: manter como
  nulo — preencher com um valor artificial (média, zero) esconderia a falha
  do sensor e distorceria análises futuras de temperatura/umidade.
- **`_flag_duplicado` (10 registros / 0,20%)**: ao inspecionar manualmente,
  nenhum par tinha valores idênticos de `duracao_segundos`, `individuos`,
  `temperatura_c` ou `umidade_pct` — ou seja, **não são duplicatas reais**.
  São dois eventos distintos da mesma espécie, na mesma câmera, caindo no
  mesmo minuto (o timestamp só tem granularidade de minuto). Decisão: manter
  os dois registros — remover um deles apagaria um evento real observado.

Nenhuma linha é removida nesta camada; os problemas ficam sinalizados nas
colunas `_flag_*` para quem for analisar decidir como tratar, caso a caso.

## Gravar tabela Silver

In [0]:
TABELA_SILVER = "fauna.monitoramento.silver_registros"

(
    df_flags.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER)
)

print(f"✅ Tabela criada: {TABELA_SILVER}")

## Validação final da Silver

In [0]:
df_check = spark.table("fauna.monitoramento.silver_registros")

print(f"Total de linhas: {df_check.count()}")
print(f"Total de colunas: {len(df_check.columns)}")

display(
    df_check.select(
        "id_registro", "id_camera", "data_hora_inicio", "duracao_segundos",
        "especie", "grupo", "individuos", "temperatura_c", "umidade_pct",
    ).limit(10)
)

## Resumo e decisões da Fase 2 — Tratamento Silver

Esta fase partiu da tabela Bronze (dado bruto) e produziu a Silver: schema
correto, valores limpos, e qualidade validada — sem enriquecimento de negócio
ainda (área, período do dia ficam pra Gold, na Fase 3).

**Decisões técnicas tomadas e por quê:**

- **`regexp_extract` em vez de `replace` de string fixa** para limpar
  `temperatura_c` e `umidade_pct`. O padrão `(-?\d+\.?\d*)` captura só a parte
  numérica, o que é mais robusto do que remover um sufixo fixo tipo `" °C"` —
  continua funcionando mesmo se o espaçamento variar ou a unidade vier diferente.

- **Checagem de nulos antes/depois da conversão.** Isso separou dois problemas
  que pareceriam iguais à primeira vista: nulo que já existia no dado bruto
  (falha real de sensor) vs. nulo que a nossa própria conversão teria causado
  (erro de parsing). Confirmamos que os 247 nulos em `temperatura_c` e 175 em
  `umidade_pct` já existiam desde a Bronze — não foram introduzidos por nós.

- **Flags separadas para "ausente" e "fora da faixa".** Em vez de uma única
  flag genérica de "valor inválido", separamos a causa: ausência de leitura
  (sensor não registrou) é um problema diferente de leitura presente mas fora
  do fisicamente plausível (0–100% para umidade, -5°C a 45°C para temperatura,
  esta última uma referência nossa, não um limite oficial).

- **Duplicidade investigada antes de decidir, não descartada de cara.** A
  flag de duplicidade (câmera + horário + espécie) sinalizou 10 registros, mas
  a inspeção manual mostrou que nenhum par tinha valores idênticos de
  `duracao_segundos`, `individuos`, `temperatura_c` ou `umidade_pct` — ou seja,
  não eram duplicatas de ingestão, e sim dois eventos reais e distintos
  coincidindo no mesmo minuto (limitação da granularidade do timestamp, não um
  erro de dado). Por isso nenhum dos dois registros de cada par foi removido.

- **Nenhum registro descartado na Silver.** Todos os problemas encontrados
  ficam marcados nas colunas `_flag_camera_invalida`, `_flag_individuos_invalido`,
  `_flag_duracao_invalida`, `_flag_temperatura_ausente`,
  `_flag_temperatura_fora_faixa`, `_flag_umidade_ausente`,
  `_flag_umidade_fora_faixa` e `_flag_duplicado` — quem decide filtrar ou não,
  em cada análise, é a Fase 4, dependendo da pergunta.

**Resultado:** tabela `fauna.monitoramento.silver_registros`, 5.000 registros,
`temperatura_c`/`umidade_pct` como `double`, `data_hora_inicio` como
`timestamp`. Nenhuma inconsistência de